# MobileNetV3 — ограниченный Optuna search

Поиск изолирован от registry: он сохраняет SQLite, таблицу trials, лучший tuning-checkpoint и YAML для отдельного полного подтверждения на Google Drive.

In [ ]:
REPO_URL = "https://github.com/frest1ler/text-orientation-classification.git"
BRANCH = "main"
PROJECT_DIR = "/content/drive/MyDrive/text-orientation"
SMOKE_RUN = True  # сначала True: максимум 2 коротких trial
N_TRIALS = 10     # используется в основном поиске; рекомендуемый диапазон 8–12
TRAIN_BASE_SAMPLES = 8000
VALIDATION_BASE_SAMPLES = 5000
FINETUNE_EPOCHS = 3
TRAIN_BATCH_SIZE = 64
VALIDATION_BATCH_SIZE = 128
NUM_WORKERS = 2
RUN_TESTS = True
RUN_FULL_CONFIRMATION = False  # включать только после анализа основного search

In [ ]:
import os, subprocess, sys
from pathlib import Path

repo_dir = Path("/content/text-orientation-classification")
if not repo_dir.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(repo_dir)], check=True)
os.chdir(repo_dir)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("В Colab выберите Runtime → Change runtime type → GPU")
print("gpu:", torch.cuda.get_device_name(0))
if RUN_TESTS:
    subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)

In [ ]:
command = [
    sys.executable, "-m", "scripts.optuna_search",
    "--project-dir", PROJECT_DIR,
    "--trials", str(N_TRIALS),
    "--train-base-samples", str(TRAIN_BASE_SAMPLES),
    "--validation-base-samples", str(VALIDATION_BASE_SAMPLES),
    "--finetune-epochs", str(FINETUNE_EPOCHS),
    "--batch-size", str(TRAIN_BATCH_SIZE),
    "--validation-batch-size", str(VALIDATION_BATCH_SIZE),
    "--num-workers", str(NUM_WORKERS),
]
if SMOKE_RUN:
    command.append("--smoke")
subprocess.run(command, check=True)

In [ ]:
import json

mode = "smoke" if SMOKE_RUN else "search"
result_dir = Path(PROJECT_DIR) / "tuning" / "optuna" / "mobilenet_v3_large" / mode
summary = json.loads((result_dir / "summary.json").read_text())
print(json.dumps(summary, indent=2))
print("Результаты:", result_dir)
print("Registry не изменён. Для полного подтверждения используйте:", result_dir / "best_config.yaml")

In [ ]:
# Опциональное полное подтверждение единственного победителя. Smoke-конфиг запрещён.
if RUN_FULL_CONFIRMATION:
    if SMOKE_RUN:
        raise ValueError("Полное подтверждение разрешено только после SMOKE_RUN=False")
    import platform, shutil
    from datetime import datetime, timezone

    confirmation_config = result_dir / "best_config.yaml"
    run_name = "colab_mobilenet_optuna_full"
    run_dir = Path("artifacts/experiments") / run_name
    recovery_dir = Path(PROJECT_DIR) / "training/recovery/mobilenet_optuna/full"
    subprocess.run([
        sys.executable, "-m", "scripts.train",
        "--config", str(confirmation_config),
        "--run-name", run_name,
        "--recovery-dir", str(recovery_dir),
        "--resume",
    ], check=True)
    environment = {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "python": platform.python_version(),
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "gpu": torch.cuda.get_device_name(0),
        "git_commit": subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(),
    }
    (run_dir / "environment.json").write_text(json.dumps(environment, indent=2), encoding="utf-8")
    subprocess.run([sys.executable, "-m", "scripts.calibrate", "--run-dir", str(run_dir), "--config", str(confirmation_config)], check=True)
    subprocess.run([sys.executable, "-m", "scripts.promote_champion", "--run-dir", str(run_dir), "--registry-dir", str(Path(PROJECT_DIR) / "registry")], check=True)
    archive = Path(shutil.make_archive("/content/colab_mobilenet_optuna_full", "zip", root_dir=run_dir))
    destination = Path(PROJECT_DIR) / "training/runs/mobilenet_v3_large/optuna_confirmation" / archive.name
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(archive, destination)
    print("Полное подтверждение сохранено:", destination)
else:
    print("Полное подтверждение пропущено: RUN_FULL_CONFIRMATION=False")